In [141]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
df = pd.read_csv("../data/social_posts.csv")

pivot_table	long → wide (with a calculation)
crosstab	long → wide (counts of combinations)
melt	wide → long

In [142]:
df.pivot_table(index="brand", columns="platform", values="likes", aggfunc="mean").round(1)

platform,Facebook,Instagram,TikTok
brand,,,
FitFuel,557.7,1069.7,1630.3
Glowly,836.3,1527.5,2505.6
HomeNest,168.6,454.9,698.2
SnackBox,806.6,2141.9,4323.3
UrbanThread,268.3,607.3,1325.9


index=	what goes down the side?	brand
columns=	what goes across the top?	platform
values=	which column to calculate on?	likes
aggfunc=	what calculation?	"mean", "sum", "count", "max"

In [143]:
p = df.pivot_table(index="brand", columns="platform", values="likes", aggfunc="mean").round(0)
p.loc["Glowly", "TikTok"]     # 2506.0
p["TikTok"]                   # one column → every brand's TikTok average
p.loc["SnackBox"]             # one row → SnackBox on each platform

np.float64(2506.0)

brand
FitFuel        1630.0
Glowly         2506.0
HomeNest        698.0
SnackBox       4323.0
UrbanThread    1326.0
Name: TikTok, dtype: float64

platform
Facebook      807.0
Instagram    2142.0
TikTok       4323.0
Name: SnackBox, dtype: float64

In [144]:
p
p.idxmax(axis=1)     # for each brand, which platform has the highest value

platform,Facebook,Instagram,TikTok
brand,,,
FitFuel,558.0,1070.0,1630.0
Glowly,836.0,1528.0,2506.0
HomeNest,169.0,455.0,698.0
SnackBox,807.0,2142.0,4323.0
UrbanThread,268.0,607.0,1326.0


brand
FitFuel        TikTok
Glowly         TikTok
HomeNest       TikTok
SnackBox       TikTok
UrbanThread    TikTok
dtype: str

In [145]:
df.pivot_table(index="brand", columns="platform", values="likes",
               aggfunc="mean", margins=True) #margins adds totals at end

platform,Facebook,Instagram,TikTok,All
brand,,,,
FitFuel,557.727273,1069.666667,1630.333333,1179.357143
Glowly,836.285714,1527.548387,2505.588235,1639.629032
HomeNest,168.625000,454.933333,698.150000,459.227273
SnackBox,806.615385,2141.875000,4323.304348,2688.783333
UrbanThread,268.300000,607.304348,1325.913043,841.910714
All,526.718750,1151.871212,2122.375000,1354.946667


fill_value=0: if a combination has no rows (e.g. a brand never posted on Facebook), that cell would be NaN. fill_value=0 puts 0 there instead. Only use this when 0 makes sense (counts, sums), not for means. A mean of “no posts” isn’t 0, it’s unknown.

In [146]:
pd.crosstab(df["brand"], df["post_type"])
# pd.crosstab(rows_column, columns_column). You pass two columns, not column names.

post_type,carousel,image,video
brand,,,
FitFuel,13,24,19
Glowly,12,22,28
HomeNest,17,30,19
SnackBox,19,23,18
UrbanThread,10,21,25


In [147]:
pd.crosstab(df["brand"], df["post_type"], normalize="index").round(2)

# normalize=	Each … adds to 1
# "index"	row
# "columns"	column
# "all"	whole table

post_type,carousel,image,video
brand,,,
FitFuel,0.23,0.43,0.34
Glowly,0.19,0.35,0.45
HomeNest,0.26,0.45,0.29
SnackBox,0.32,0.38,0.30
UrbanThread,0.18,0.38,0.45


In [148]:
small = df[["post_id", "likes", "comments", "shares"]]
long = small.melt(id_vars="post_id", var_name="metric", value_name="count")

In [149]:
# id_vars=	column(s) to keep as they are (they repeat on every row)
# var_name=	name for the new column holding the old column names
# value_name=	name for the new column holding the values

In [150]:
p.reset_index().melt(id_vars="brand", var_name="platform", value_name="avg_likes")

,brand,platform,avg_likes
0,FitFuel,Facebook,558.0
1,Glowly,Facebook,836.0
2,HomeNest,Facebook,169.0
3,SnackBox,Facebook,807.0
4,UrbanThread,Facebook,268.0
5,FitFuel,Instagram,1070.0
6,Glowly,Instagram,1528.0
7,HomeNest,Instagram,455.0
8,SnackBox,Instagram,2142.0
9,UrbanThread,Instagram,607.0


## 2.8 Cheat sheet: Reshaping

| I want… | Write |
|---|---|
| grid of a stat (row group × column group) | `df.pivot_table(index="r", columns="c", values="v", aggfunc="mean")` |
| add totals row/column | `margins=True` |
| empty combos as 0 (counts/sums only) | `fill_value=0` |
| one cell of a pivot | `p.loc["row_label", "col_name"]` |
| best column for each row | `p.idxmax(axis=1)` |
| count combinations | `pd.crosstab(df["r"], df["c"])` |
| row proportions | `pd.crosstab(df["r"], df["c"], normalize="index")` |
| wide → long | `df.melt(id_vars="id", var_name="name", value_name="value")` |
| pivot → long again | `p.reset_index().melt(id_vars="r")` |

**Rules**
- pivot_table = groupby with 2 keys, shown as a grid
- crosstab takes two columns (Series), not names
- melt rows = rows × number of melted columns
- fill_value=0 only when 0 is a real answer (not for means)

In [151]:
sales = pd.DataFrame({
    "shop":  ["A", "A", "B", "B", "A"],
    "day":   ["Mon", "Tue", "Mon", "Tue", "Mon"],
    "units": [5, 3, 4, 6, 7],
})

In [152]:
piv= sales.pivot_table(index="shop", columns="day", values="units", aggfunc="sum")
# day	Mon	Tue
# shop		
# A	12	3
# B	4	6

In [153]:
sales.pivot_table(index="shop", columns="day", values="units", aggfunc="mean")
# day	Mon	Tue
# shop		
# A	6.0	3.0
# B	4.0	6.0

day,Mon,Tue
shop,,
A,6.0,3.0
B,4.0,6.0


In [154]:
# .shape of the pivot in Q1.
piv.shape 
#2,2

(2, 2)

In [155]:
pd.crosstab(sales["shop"], sales["day"])
# day	Mon	Tue
# shop		
# A	2	1
# B	1	1

day,Mon,Tue
shop,,
A,2,1
B,1,1


In [156]:
pd.crosstab(sales["shop"], sales["day"], normalize="index")
# day	Mon	Tue
# shop		
# A	0.667	0.333

day,Mon,Tue
shop,,
A,0.666667,0.333333
B,0.500000,0.500000


In [157]:
sales.pivot_table(index="shop", columns="day", values="units", aggfunc="sum", margins= True)
#3,3
#all/all gives total of all sales of both shops and the week

day,Mon,Tue,All
shop,,,
A,12,3,15
B,4,6,10
All,16,9,25


In [158]:
wide = pd.DataFrame({"shop": ["A", "B"], "Mon": [12, 4], "Tue": [3, 6]})
wide.melt(id_vars="shop", var_name= 'day', value_name='sale')
wide.melt(id_vars='shop').shape
#4,3-- defaults are variable and value

,shop,day,sale
0,A,Mon,12
1,B,Mon,4
2,A,Tue,3
3,B,Tue,6


(4, 3)

In [159]:
piv.loc["B", "Tue"] #6
piv.idxmax(axis=1)

# A    Mon
# B    Tue

np.int64(6)

shop
A    Mon
B    Tue
dtype: str

In [160]:
# B1. Likes grid

# (a) Pivot table: brand down the side, platform across the top, mean likes.
p=df.pivot_table(index= 'brand', columns= 'platform', values= 'likes', aggfunc= 'mean').round(0)
# (b) Round to 0.
# (c) Store it as p.
p
# (d) Get Glowly’s TikTok value using .loc.
p.loc['Glowly', 'TikTok']
# (e) Use idxmax(axis=1) to find each brand’s best platform.
p.idxmax(axis=1)
# (f) Comment: what pattern do you see in (e)?
# TikTok is the leading platform for each brand.

platform,Facebook,Instagram,TikTok
brand,,,
FitFuel,558.0,1070.0,1630.0
Glowly,836.0,1528.0,2506.0
HomeNest,169.0,455.0,698.0
SnackBox,807.0,2142.0,4323.0
UrbanThread,268.0,607.0,1326.0


np.float64(2506.0)

brand
FitFuel        TikTok
Glowly         TikTok
HomeNest       TikTok
SnackBox       TikTok
UrbanThread    TikTok
dtype: str

In [161]:
# B2. With totals

# (a) Redo B1’s pivot with margins=True.
p_m=df.pivot_table(index= 'brand', columns= 'platform', values= 'likes', aggfunc= 'mean', margins=True).round(0)
# (b) Round to 0.
p_m
# (c) Comment: what does the All/All cell mean? Check it against df["likes"].mean().
df["likes"].mean() #gives the general mean of the all likes

platform,Facebook,Instagram,TikTok,All
brand,,,,
FitFuel,558.0,1070.0,1630.0,1179.0
Glowly,836.0,1528.0,2506.0,1640.0
HomeNest,169.0,455.0,698.0,459.0
SnackBox,807.0,2142.0,4323.0,2689.0
UrbanThread,268.0,607.0,1326.0,842.0
All,527.0,1152.0,2122.0,1355.0


np.float64(1354.9466666666667)

In [162]:
# B3. Post mix per brand

# (a) Crosstab of brand × post_type (counts).
pd.crosstab(df['brand'], df['post_type'])
# (b) Redo it with normalize="index".
pd.crosstab(df['brand'], df['post_type'], normalize="index").round(2)
# (c) Round to 2.
# (d) Comment: which two brands rely most on video?
#Glowly, UrbanThread
# (e) Comment: which brand posts the most carousels proportionally?
#Snackbox

post_type,carousel,image,video
brand,,,
FitFuel,13,24,19
Glowly,12,22,28
HomeNest,17,30,19
SnackBox,19,23,18
UrbanThread,10,21,25


post_type,carousel,image,video
brand,,,
FitFuel,0.23,0.43,0.34
Glowly,0.19,0.35,0.45
HomeNest,0.26,0.45,0.29
SnackBox,0.32,0.38,0.30
UrbanThread,0.18,0.38,0.45


In [163]:
# B4. Sponsored share by platform

# (a) Crosstab of platform × sponsored with normalize="index".
pd.crosstab(df['platform'], df['sponsored'], normalize='index').round(3)
# (b) Round to 3.
# (c) Comment: which platform has the highest share of sponsored posts?
#Facebook

sponsored,False,True
platform,,
Facebook,0.703,0.297
Instagram,0.758,0.242
TikTok,0.788,0.212


In [164]:
# B5. Same as 2.6 B9, as a grid

# (a) Pivot table: brand down the side, sponsored across the top, mean likes.
df.pivot_table(index= 'brand', columns= 'sponsored', values='likes', aggfunc='mean').round(1)
df.groupby(['brand', 'sponsored'])['likes'].mean().round(1)
# (b) Round to 1.
# (c) Comment: compare this to your 2.6 B9 output. What’s the same, and which is easier to read?
#pivot is more well-organized, values are the same. 

sponsored,False,True
brand,,
FitFuel,1150.9,1250.5
Glowly,1580.2,1915.1
HomeNest,487.6,389.1
SnackBox,2936.8,1792.0
UrbanThread,850.1,817.4


brand        sponsored
FitFuel      False        1150.9
             True         1250.5
Glowly       False        1580.2
             True         1915.1
HomeNest     False         487.6
             True          389.1
SnackBox     False        2936.8
             True         1792.0
UrbanThread  False         850.1
             True          817.4
Name: likes, dtype: float64

In [165]:
# B6. Counting with pivot_table

# (a) Pivot table: platform down the side, post_type across the top, values="post_id", aggfunc="count".
df.pivot_table(index='platform', columns= 'post_type', values='post_id', aggfunc='count')
# (b) Comment: which crosstab gives the same table? Write the crosstab line too.
pd.crosstab(df['platform'], df['post_type'])

post_type,carousel,image,video
platform,,,
Facebook,14,27,23
Instagram,30,50,52
TikTok,27,43,34


post_type,carousel,image,video
platform,,,
Facebook,14,27,23
Instagram,30,50,52
TikTok,27,43,34


In [166]:
# B7. melt: metrics to long

# (a) Make small = post_id, likes, comments and shares only.
small= df[['post_id', 'likes', 'comments', 'shares']]
# (b) Melt it with id_vars="post_id", var_name="metric", value_name="count". Store it as long.
long=small.melt(id_vars="post_id", var_name="metric", value_name="count")
# (c) Show its shape.
long.shape
# (d) Comment: why that many rows?
#for each post, we have a separate row for likes, comments, shares , giving 3 rows per post
# (e) Total count per metric using groupby (2.6).
long.groupby("metric")["count"].sum()   # likes 406,484 · comments 15,841 · shares 7,341

(900, 3)

metric
comments     15841.0
likes       406484.0
shares        7341.0
Name: count, dtype: float64

In [167]:
# B8. Pivot back to long

# (a) Take p from B1 and .reset_index() it.
p_reset=p.reset_index()
# (b) Melt it with id_vars="brand", var_name="platform", value_name="avg_likes". Store it as back.
back= p_reset.melt(id_vars="brand", var_name="platform", value_name="avg_likes")
# (c) Show its shape.
back.shape, back
# (d) Sort by avg_likes, highest first, and show the top 3 rows.
back.sort_values('avg_likes', ascending=False).head(3)

((15, 3),
           brand   platform  avg_likes
 0       FitFuel   Facebook      558.0
 1        Glowly   Facebook      836.0
 2      HomeNest   Facebook      169.0
 3      SnackBox   Facebook      807.0
 4   UrbanThread   Facebook      268.0
 5       FitFuel  Instagram     1070.0
 6        Glowly  Instagram     1528.0
 7      HomeNest  Instagram      455.0
 8      SnackBox  Instagram     2142.0
 9   UrbanThread  Instagram      607.0
 10      FitFuel     TikTok     1630.0
 11       Glowly     TikTok     2506.0
 12     HomeNest     TikTok      698.0
 13     SnackBox     TikTok     4323.0
 14  UrbanThread     TikTok     1326.0)

,brand,platform,avg_likes
13,SnackBox,TikTok,4323.0
11,Glowly,TikTok,2506.0
8,SnackBox,Instagram,2142.0


In [168]:
pd.crosstab(df["brand"], df["platform"]) #input has to be the full column, not just name

platform,Facebook,Instagram,TikTok
brand,,,
FitFuel,11,24,21
Glowly,14,31,17
HomeNest,16,30,20
SnackBox,13,24,23
UrbanThread,10,23,23


In [169]:
df.pivot_table(index="brand", columns="platform", values="likes", aggfunc='sum')
#did not define sum

platform,Facebook,Instagram,TikTok
brand,,,
FitFuel,6135,25672,34237
Glowly,11708,47354,42595
HomeNest,2698,13648,13963
SnackBox,10486,51405,99436
UrbanThread,2683,13968,30496


In [171]:
df.pivot_table(index="brand", columns="platform", values="likes", aggfunc="mean", fill_value=0)
# The idea isn’t that NaN affects the mean. It’s this: if a brand has no posts on a platform, its average is unknown. Putting 0 there claims “their posts averaged zero likes”, which is false and makes that brand look terrible. Use 0 for counts and sums (“no posts = 0 posts” is true), never for means.

platform,Facebook,Instagram,TikTok
brand,,,
FitFuel,557.727273,1069.666667,1630.333333
Glowly,836.285714,1527.548387,2505.588235
HomeNest,168.625000,454.933333,698.150000
SnackBox,806.615385,2141.875000,4323.304348
UrbanThread,268.300000,607.304348,1325.913043
